# 01 — EDA, Time-Ordered Split & Class Imbalance Analysis

**Project**: UPI Fraud Detection — Drift-Aware Explainable ML Framework
**Datasets**: IEEE-CIS Fraud Detection (primary), PaySim (secondary)
**Goal**: Understand data, create time-ordered splits, analyze class imbalance, prepare for baseline modeling

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from data_prep import load_ieee_cis, load_paysim, time_ordered_split, prepare_features_target, handle_class_imbalance

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_ROOT = Path('../data')
print(f"Data root: {DATA_ROOT.absolute()}")

In [ ]:
# Load IEEE-CIS
try:
    ieee_df = load_ieee_cis(DATA_ROOT / 'ieee-cis')
    print(f"IEEE-CIS loaded: {ieee_df.shape}")
    print(f"Time range: {ieee_df['TransactionDT'].min()} to {ieee_df['TransactionDT'].max()}")
    print(f"Fraud rate: {ieee_df['isFraud'].mean():.4f}")
    display(ieee_df.head())
except FileNotFoundError as e:
    print(f"IEEE-CIS not found: {e}")
    ieee_df = None

In [ ]:
# Load PaySim
try:
    paysim_df = load_paysim(DATA_ROOT / 'paysim')
    print(f"PaySim loaded: {paysim_df.shape}")
    print(f"Time range (steps): {paysim_df['step'].min()} to {paysim_df['step'].max()}")
    print(f"Fraud rate: {paysim_df['isFraud'].mean():.4f}")
    display(paysim_df.head())
except FileNotFoundError as e:
    print(f"PaySim not found: {e}")
    paysim_df = None

## 1. IEEE-CIS Exploratory Data Analysis

In [ ]:
if ieee_df is not None:
    print("=== Shape ===", ieee_df.shape)
    print("=== dtypes ===")
    print(ieee_df.dtypes.value_counts())
    print("=== Missing values ===")
    missing = ieee_df.isnull().sum().sort_values(ascending=False)
    print(missing[missing > 0].head(20))
    print(f"Total missing: {missing.sum()}")
    print("=== Fraud distribution ===")
    print(ieee_df['isFraud'].value_counts())
    print(f"Fraud rate: {ieee_df['isFraud'].mean():.4%}")

In [ ]:
if ieee_df is not None:
    # Transaction amount by fraud
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Log amount distribution
    for label, color in [(0, 'blue'), (1, 'red')]:
        subset = ieee_df[ieee_df['isFraud'] == label]['TransactionAmt']
        axes[0].hist(np.log1p(subset), bins=50, alpha=0.5, label=f'Fraud={label}', color=color, density=True)
    axes[0].set_xlabel('log(TransactionAmt + 1)')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Transaction Amount Distribution by Fraud Label')
    axes[0].legend()
    
    # Fraud rate over time (TransactionDT)
    ieee_df['day'] = (ieee_df['TransactionDT'] // 86400).astype(int)
    daily_fraud = ieee_df.groupby('day')['isFraud'].agg(['mean', 'count']).reset_index()
    daily_fraud.columns = ['day', 'fraud_rate', 'count']
    axes[1].plot(daily_fraud['day'], daily_fraud['fraud_rate'], marker='o', markersize=3)
    axes[1].set_xlabel('Day (since start)')
    axes[1].set_ylabel('Fraud Rate')
    axes[1].set_title('Daily Fraud Rate Over Time (Concept Drift Check)')
    
    plt.tight_layout()
    plt.show()

In [ ]:
if ieee_df is not None:
    # Categorical feature fraud rates
    cat_cols = ieee_df.select_dtypes(include=['category']).columns.tolist()
    if 'isFraud' in cat_cols:
        cat_cols.remove('isFraud')
    
    print(f"Categorical columns ({len(cat_cols)}): {cat_cols[:10]}...")
    
    # Top categories by fraud rate for first few cat columns
    for col in cat_cols[:5]:
        fraud_by_cat = ieee_df.groupby(col)['isFraud'].agg(['mean', 'count']).reset_index()
        fraud_by_cat = fraud_by_cat[fraud_by_cat['count'] > 100].sort_values('mean', ascending=False)
        print(f"\n{col} - Top 10 by fraud rate:")
        print(fraud_by_cat.head(10).to_string(index=False))

In [ ]:
if ieee_df is not None:
    # Numerical feature correlations with fraud
    num_cols = ieee_df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c not in ['TransactionID', 'TransactionDT', 'isFraud', 'day']]
    
    correlations = ieee_df[num_cols + ['isFraud']].corr()['isFraud'].sort_values(ascending=False)
    print("Top 15 numerical features correlated with fraud:")
    print(correlations.head(15))
    
    # Plot top correlations
    top_corr = correlations[1:16]  # exclude isFraud itself
    plt.figure(figsize=(10, 6))
    top_corr.plot(kind='barh')
    plt.xlabel('Correlation with isFraud')
    plt.title('Top 15 Numerical Features by Fraud Correlation')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 2. PaySim Exploratory Data Analysis

In [ ]:
if paysim_df is not None:
    print("=== Shape ===", paysim_df.shape)
    print("=== dtypes ===")
    print(paysim_df.dtypes.value_counts())
    print("=== Missing values ===")
    missing = paysim_df.isnull().sum()
    print(missing[missing > 0])
    print("=== Fraud distribution ===")
    print(paysim_df['isFraud'].value_counts())
    print(f"Fraud rate: {paysim_df['isFraud'].mean():.4%}")
    print("=== Transaction types ===")
    print(paysim_df['type'].value_counts())

In [ ]:
if paysim_df is not None:
    # Fraud by transaction type
    fraud_by_type = paysim_df.groupby('type')['isFraud'].agg(['mean', 'count']).reset_index()
    fraud_by_type.columns = ['type', 'fraud_rate', 'count']
    print(fraud_by_type.to_string(index=False))
    
    # Amount distribution by fraud
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for label, color in [(0, 'blue'), (1, 'red')]:
        subset = paysim_df[paysim_df['isFraud'] == label]['amount']
        axes[0].hist(np.log1p(subset), bins=50, alpha=0.5, label=f'Fraud={label}', color=color, density=True)
    axes[0].set_xlabel('log(Amount + 1)')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Amount Distribution by Fraud Label')
    axes[0].legend()
    
    # Fraud rate over steps (time)
    step_fraud = paysim_df.groupby('step')['isFraud'].agg(['mean', 'count']).reset_index()
    step_fraud.columns = ['step', 'fraud_rate', 'count']
    axes[1].plot(step_fraud['step'], step_fraud['fraud_rate'], marker='o', markersize=2)
    axes[1].set_xlabel('Step (Hour)')
    axes[1].set_ylabel('Fraud Rate')
    axes[1].set_title('Hourly Fraud Rate Over Time')
    
    plt.tight_layout()
    plt.show()

## 3. Time-Ordered Train/Val/Test Split

In [ ]:
# Split IEEE-CIS
if ieee_df is not None:
    print("=== IEEE-CIS Time-Ordered Split (70/15/15) ===")
    train_ieee, val_ieee, test_ieee = time_ordered_split(ieee_df, 'TransactionDT')
    
    # Save splits
    split_dir = DATA_ROOT / 'ieee-cis' / 'splits'
    split_dir.mkdir(parents=True, exist_ok=True)
    train_ieee.to_parquet(split_dir / 'train.parquet', index=False)
    val_ieee.to_parquet(split_dir / 'val.parquet', index=False)
    test_ieee.to_parquet(split_dir / 'test.parquet', index=False)
    print(f"Splits saved to {split_dir}")

In [ ]:
# Split PaySim
if paysim_df is not None:
    print("=== PaySim Time-Ordered Split (70/15/15) ===")
    train_paysim, val_paysim, test_paysim = time_ordered_split(paysim_df, 'step')
    
    split_dir = DATA_ROOT / 'paysim' / 'splits'
    split_dir.mkdir(parents=True, exist_ok=True)
    train_paysim.to_parquet(split_dir / 'train.parquet', index=False)
    val_paysim.to_parquet(split_dir / 'val.parquet', index=False)
    test_paysim.to_parquet(split_dir / 'test.parquet', index=False)
    print(f"Splits saved to {split_dir}")

## 4. Class Imbalance Analysis & Handling Strategy

In [ ]:
def analyze_imbalance(y, name):
    """Analyze class imbalance and recommend handling."""
    fraud_rate = y.mean()
    n_fraud = y.sum()
    n_normal = len(y) - n_fraud
    ratio = n_normal / n_fraud if n_fraud > 0 else np.inf
    
    print(f"=== {name} Class Imbalance ===")
    print(f"  Total: {len(y):,}")
    print(f"  Fraud: {n_fraud:,} ({fraud_rate:.4%})")
    print(f"  Normal: {n_normal:,} ({1-fraud_rate:.4%})")
    print(f"  Imbalance ratio (normal:fraud): {ratio:.1f}:1")
    
    # Recommendations
    if ratio > 100:
        print(f"  → EXTREME imbalance: Use class_weight + consider SMOTE/ADASYN")
    elif ratio > 20:
        print(f"  → HIGH imbalance: Use class_weight (scale_pos_weight={ratio:.1f})")
    elif ratio > 5:
        print(f"  → MODERATE imbalance: class_weight recommended")
    else:
        print(f"  → Low imbalance: Standard training OK")
    
    return fraud_rate, ratio

if ieee_df is not None:
    X_train, y_train = prepare_features_target(train_ieee)
    analyze_imbalance(y_train, "IEEE-CIS Train")
    
    # Get class_weight params
    params = handle_class_imbalance(X_train, y_train, method='class_weight')
    print(f"\nRecommended XGBoost/LightGBM params: {params}")

In [ ]:
if paysim_df is not None:
    X_train_p, y_train_p = prepare_features_target(train_paysim)
    analyze_imbalance(y_train_p, "PaySim Train")
    
    params_p = handle_class_imbalance(X_train_p, y_train_p, method='class_weight')
    print(f"\nRecommended XGBoost/LightGBM params: {params_p}")

## 5. Feature Engineering Preview

In [ ]:
def basic_feature_engineering(df, dataset_name):
    """Apply basic feature engineering."""
    df = df.copy()
    
    if dataset_name == 'ieee-cis':
        # Time features from TransactionDT (seconds since epoch)
        df['hour'] = (df['TransactionDT'] // 3600) % 24
        df['day_of_week'] = (df['TransactionDT'] // 86400) % 7
        
        # Amount features
        df['amount_log'] = np.log1p(df['TransactionAmt'])
        df['amount_round'] = (df['TransactionAmt'] == df['TransactionAmt'].round()).astype(int)
        
        # Card features
        if 'card1' in df.columns:
            df['card1_freq'] = df.groupby('card1')['card1'].transform('count')
    
    elif dataset_name == 'paysim':
        # Time features from step (hours)
        df['hour'] = df['step'] % 24
        df['day'] = df['step'] // 24
        
        # Amount features
        df['amount_log'] = np.log1p(df['amount'])
        df['amount_round'] = (df['amount'] == df['amount'].round()).astype(int)
        
        # Balance change features
        df['orig_balance_diff'] = df['oldbalanceOrg'] - df['newbalanceOrig']
        df['dest_balance_diff'] = df['newbalanceDest'] - df['oldbalanceDest']
        df['zero_balance_orig'] = (df['oldbalanceOrg'] == 0).astype(int)
        df['zero_balance_dest'] = (df['oldbalanceDest'] == 0).astype(int)
    
    return df

if ieee_df is not None:
    train_ieee_fe = basic_feature_engineering(train_ieee, 'ieee-cis')
    print(f"Engineered features: {train_ieee_fe.shape[1] - train_ieee.shape[1]} new columns")
    print(f"New columns: {[c for c in train_ieee_fe.columns if c not in train_ieee.columns]}")

In [ ]:
if paysim_df is not None:
    train_paysim_fe = basic_feature_engineering(train_paysim, 'paysim')
    print(f"Engineered features: {train_paysim_fe.shape[1] - train_paysim.shape[1]} new columns")
    print(f"New columns: {[c for c in train_paysim_fe.columns if c not in train_paysim.columns]}")

## 6. Summary & Next Steps

In [ ]:
print("""
=== EDA SUMMARY ===

✅ Data loaded and validated
✅ Time-ordered splits created (70/15/15)
✅ Class imbalance analyzed
✅ Basic feature engineering applied

NEXT: Notebook 02_baselines.ipynb
- Train Logistic Regression, Random Forest, XGBoost, LightGBM
- Rolling time-window cross-validation
- Evaluate: PR-AUC, Precision@K, Recall@K
- Log results for comparison
""")